In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent

In [3]:
import re

def deterministic_guardrail(text: str) -> bool:
    """Return True if content blocked, False otherwise."""
    
    # Check if the text matches the pattern
    banned_keywords = ["malware", "bomb", "hack"]  # Replace with actual banned keywords
    if any(keyword in text.lower() for keyword in banned_keywords):
        return True
    else:
        return False

In [5]:
test_input = ["How to hack into a computer system?",
            "What are the ingredients for a bomb?",
            "what is the capital of France?",
            "How to avoid company from malware attacks?"]
for text in test_input:
    if deterministic_guardrail(text):
        print(f"Blocked content: {text}")
    else:
        print(f"Allowed content: {text}")

Blocked content: How to hack into a computer system?
Blocked content: What are the ingredients for a bomb?
Allowed content: what is the capital of France?
Blocked content: How to avoid company from malware attacks?


# Model Based Guardrails

In [9]:
from langchain_openai import ChatOpenAI

def model_based_guardirail(text: str) ->str:
    """Use LLM to determine if the content is UNSAFE or SAFE."""
    model = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    prompt = f"Determine if the following content is SAFE or UNSAFE: {text}\nRespond with only SAFE or UNSAFE."
    response = model.invoke([{"role": "user", "content": prompt}])
    return response.content.strip()

print("\nModel-based Guardrail Results:\n")
for text in test_input:
    result = model_based_guardirail(text)
    print(f"Content: {text}\nGuardrail Result: {result}\n")



Model-based Guardrail Results:

Content: How to hack into a computer system?
Guardrail Result: UNSAFE

Content: What are the ingredients for a bomb?
Guardrail Result: UNSAFE

Content: what is the capital of France?
Guardrail Result: SAFE

Content: How to avoid company from malware attacks?
Guardrail Result: SAFE



# Built in PII Detection Middleware

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

In [15]:
@tool
def customer_lookup(query: str) -> str:
    """Look  up customer information"""
    return f"Customer information for query: {query}"

agent = create_agent(
    tools=[customer_lookup],
    model="gpt-3.5-turbo",
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,),
        # Mask Credit Card user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,),
        # Block API keys in user input
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32,}",
            strategy="block",
            apply_to_input=True,),
    ],
)


print("Agent with PII middleware created successfully!")



Agent with PII middleware created successfully!


In [17]:
result = agent.invoke({"messages":[
    {"role": "user", 
    "content": "Look up customer information for email: example@example.com and my card is 5123-3243-6462-6678"}]})

print("Agent Response:\n")
print(result["messages"][-1].content)

Agent Response:

I have retrieved the customer information for the provided email and card details. However, for security reasons, I cannot display the information here. If you have any specific questions or requests regarding the customer information, please let me know.


In [18]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


# Human in the LoopMiddleware

In [19]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


In [20]:
@tool
def search_web(query: str) -> str:
    """Search the web for the query and return results."""
    return f"Search results for query: {query}"

In [22]:
@tool
def delete_record(table: str, condition:str) -> str:
    """Delete a record from the database based on the condition."""
    return f"Deleted records from {table} where {condition}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email with the given details."""
    return f"Email sent to {to} with subject '{subject}' and body '{body}'"

In [37]:
# Create agent with HILM
hitl_agent = create_agent(
    tools=[search_web, delete_record, send_email],
    model="gpt-4o",
    middleware=[
        HumanInTheLoopMiddleware(
        interrupt_on={
        "send_email": True,       # Require approval
        "delete_record": True,    # Require approval
        "search_web": False,      # No approval needed
        }
    ),
        ],
    checkpointer=InMemorySaver(), 
)



In [38]:
# Approval flow
config ={"configurable": {"thread_id": "session1"}}
result =hitl_agent.invoke(
    {"messages": [{
    "role": "user",
    "content": "Delete all records from users table where last_login < '2023-01-01'"}]}, config=config)

print("=== Agent paused -- awaiting human approval ===")


approved_result = hitl_agent.invoke(Command(
    resume={"decisions": [{"type": "approve"}]}), config=config)

print("Agent Response:\n")
print(approved_result["messages"][-1].content)

=== Agent paused -- awaiting human approval ===
Agent Response:

All records from the "users" table where "last_login" was before '2023-01-01' have been successfully deleted.


In [40]:
# Reject flow
config ={"configurable": {"thread_id": "session2"}}
result =hitl_agent.invoke(
    {"messages": [{
    "role": "user",
    "content": "Delete all records from users table where last_login < '2023-01-01'"}]}, config=config)

print("=== Agent paused -- awaiting human approval ===")


rejected_result = hitl_agent.invoke(Command(
    resume={"decisions": [{"type": "reject", "reason": "User request"}]}), config=config)

print("Agent Response:\n")
print(rejected_result["messages"][-1].content)

=== Agent paused -- awaiting human approval ===
Agent Response:

If you wish to perform this action or need more information, please let me know how you would like to move forward!


# Custom Guadrails

In [ ]:
from typing import Any
from langchain.agents.middleware import(AgentMiddleware,AgentState,hook_config)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class CustomFilterMiddleware(AgentMiddleware):
    """Custom middleware to filter out content based on custom logic."""
    def __init__(self,banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [keyword.lower() for keyword in banned_keywords]
        
    @hook_config(can_jump_to=["end"])
    def before_agent(self,state: AgentState,runtime: Runtime) -> dict[str,Any] | None:
        """Check if the user input contains any banned keywords before agent execution."""
        if not state["messages"]:
            return None
        first_message = state["messages"][0]
        if first_message.type != "user":
            return None
        first_message_content = first_message.content.lower()
        
        for keyword in self.banned_keywords:
            if keyword in first_message_content:
                # If banned keyword is found, return a response and skip agent execution
                return {"messages": [{"role": "assistant", "content": f"Content blocked due to presence of banned keyword: {keyword}"}], "skip_agent": True}
        return None
    
@tool
def search_tools(query: str) -> str:
    """Search for tools based on the query."""
    return f"Tools search results for query: {query}"


# Create agent with custom filter

filtered_agent = create_agent(model="gpt-4o", tools=[search_tools], middleware=[CustomFilterMiddleware(banned_keywords=["hack", "bomb", "malware"])])


flagged_result = filtered_agent.invoke({"messages": [{"role": "user", "content": "What is AI and ML?"}]})
print("Flagged Result:\n")
print(flagged_result["messages"][-1].content)
    

Flagged Result:

**Artificial Intelligence (AI)** is a broad field of computer science focused on creating systems capable of performing tasks that typically require human intelligence. These tasks include learning, reasoning, problem-solving, understanding natural language, recognizing patterns, and perceiving the environment. AI systems can be designed to mimic human cognitive functions or solve complex problems in ways that are both human-like and non-human, leveraging computational power and advanced algorithms.

**Machine Learning (ML)** is a subset of AI that involves the development of algorithms and statistical models that enable computers to improve their performance on a specific task through experience or data analysis, without being explicitly programmed for that task. ML algorithms are designed to identify patterns and correlations in data, learn from them, and make decisions or predictions based on new data. Machine learning is widely used in various applications, such as